# Physical benchmark: transverse-longitudinal Ising dimer
Reproduces the interacting two-qubit benchmark and the uncertainty-threshold sweep used in the manuscript.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from scipy.optimize import minimize

I=np.eye(2,dtype=complex); X=np.array([[0,1],[1,0]],complex); Z=np.array([[1,0],[0,-1]],complex)
J,hx,hz=1.0,0.8,0.3
Ham=-J*np.kron(Z,Z)-hx*(np.kron(X,I)+np.kron(I,X))-hz*(np.kron(Z,I)+np.kron(I,Z))
evals,_=eigh(Ham); E0=evals[0]; Delta=evals[1]-evals[0]; W=evals[-1]-evals[0]
print('spectrum=',evals)

def RY(t):
    c,s=np.cos(t/2),np.sin(t/2); return np.array([[c,-s],[s,c]],complex)
def dRY(t):
    c,s=np.cos(t/2),np.sin(t/2); return .5*np.array([[-s,-c],[c,-s]],complex)
CNOT=np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]],complex)
ket00=np.array([1,0,0,0],complex)
def state(th):
    a,b,c,d=th; return np.kron(RY(c),RY(d))@CNOT@np.kron(RY(a),RY(b))@ket00
def state_derivs(th):
    a,b,c,d=th; A=np.kron(RY(a),RY(b)); B=np.kron(RY(c),RY(d)); psi=B@CNOT@A@ket00
    ds=[B@CNOT@np.kron(dRY(a),RY(b))@ket00, B@CNOT@np.kron(RY(a),dRY(b))@ket00, np.kron(dRY(c),RY(d))@CNOT@A@ket00, np.kron(RY(c),dRY(d))@CNOT@A@ket00]
    return psi,ds
def energy(th):
    psi=state(th); return float(np.real(np.vdot(psi,Ham@psi)))
rng=np.random.default_rng(20260910); best=None
for _ in range(24):
    res=minimize(energy,rng.uniform(-np.pi,np.pi,4),method='BFGS',options={'gtol':1e-12,'maxiter':2000})
    if best is None or res.fun<best.fun: best=res
print('theta*=',best.x,'energy error=',best.fun-E0)

psi,ds=state_derivs(best.x); D=[v-psi*np.vdot(psi,v) for v in ds]
Gf=np.array([[np.real(np.vdot(D[i],D[j])) for j in range(4)] for i in range(4)])
Hf=np.array([[2*np.real(np.vdot(D[i],(Ham-E0*np.eye(4))@D[j])) for j in range(4)] for i in range(4)])
wg,Ug=np.linalg.eigh(Gf); mask=wg>1e-10; U=Ug[:,mask]; G=U.T@Gf@U; Hact=U.T@Hf@U
def invsqrt(A):
    w,V=np.linalg.eigh(A); return V@np.diag(1/np.sqrt(w))@V.T
S0=invsqrt(G)@Hact@invsqrt(G)
gminus,gplus=np.linalg.eigvalsh(G)[[0,-1]]; kappaH=np.linalg.cond(Hact)
epscrit=gminus/2*((Delta/W)*kappaH-1)
print('active rank=',mask.sum(),'g-,g+=',gminus,gplus)
print('spec(S0)=',np.linalg.eigvalsh(S0),'kappa(S0)=',np.linalg.cond(S0),'W/Delta=',W/Delta)
print('kappa_H=',kappaH,'epsilon_crit=',epscrit,'commutator=',np.linalg.norm(G@Hact-Hact@G,2))

raw=np.array([[.10,-.20,.15],[-.20,-.05,.10],[.15,.10,-.08]])
direction=raw/np.linalg.norm(raw,2); eps=lam=.01; Xi=eps*direction
M=G+Xi+lam*np.eye(3); Sl=invsqrt(M)@Hact@invsqrt(M); K=(W/Delta)*(1+2*eps/gminus)
print('kappa(S_lambda)=',np.linalg.cond(Sl),'K=',K)
assert eps<epscrit and np.linalg.cond(Sl)<K<kappaH

eps_grid=np.linspace(0,.026,131); Kvals=[]; actual=[]
for e in eps_grid:
    Kvals.append((W/Delta)*(1+2*e/gminus))
    M=G+e*direction+e*np.eye(3); S=invsqrt(M)@Hact@invsqrt(M); actual.append(np.linalg.cond(S))
plt.figure(figsize=(7.2,4.8))
plt.plot(eps_grid,Kvals,label=r'Theorem bound $K(\varepsilon,\varepsilon)$')
plt.plot(eps_grid,actual,label=r'Actual $\kappa(S_\lambda)$')
plt.axhline(kappaH,ls='--',label=r'Euclidean $\kappa_H$')
plt.axvline(epscrit,ls=':',label=r'$\varepsilon_{\rm crit}$')
plt.xlabel(r'Metric-error radius $\varepsilon$ with $\lambda=\varepsilon$'); plt.ylabel('Condition number'); plt.legend(); plt.tight_layout(); plt.show()
